# ScienceQA Anhedonia Results — Analysis & Visualisation

Analyses `results_scienceqa/raw_responses.csv` produced by `eval.py`.

**Design:** 5 runs × 100 prompts × 2 tiers = 1,000 forward passes. Latin square position balance. Points: [10, 20, 30, 40].

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'grid.linestyle':   '--',
    'figure.dpi':       130,
})

# ── Colours ──────────────────────────────────────────────
BLUE   = '#378ADD'
CORAL  = '#D85A30'
GRAY   = '#888780'
PT_COLORS = {10: '#B5D4F4', 20: '#FFB74D', 30: '#42A5F5', 40: '#EF5350'}
POINTS = [10, 20, 30, 40]

# ── Load ─────────────────────────────────────────────────
RAW_PATH = 'results_scienceqa/raw_responses.csv'
df = pd.read_csv(RAW_PATH)
df['points'] = pd.to_numeric(df['points'], errors='coerce')

df_clean = df[~df['collapsed']].dropna(subset=['points']).copy()
base = df_clean[df_clean['tier'] == 'baseline']
ablated = df_clean[df_clean['tier'] == 'model_A']

print(f'Total rows       : {len(df):,}')
print(f'baseline  valid  : {len(base)}  (collapsed: {df[df["tier"]=="baseline"]["collapsed"].sum()})')
print(f'model_A   valid  : {len(ablated)}  (collapsed: {df[df["tier"]=="model_A"]["collapsed"].sum()} = {df[df["tier"]=="model_A"]["collapsed"].mean()*100:.1f}%)')

## 1 — Core metric: mean points chosen ± SEM

Primary behavioural signal. SEM computed over 5 random 80-question subsets to account for within-run correlation.

In [ ]:
# ── SEM via subsets ───────────────────────────────────────
rng = np.random.default_rng(42)
ids = df['id'].unique().tolist()

def subset_sem(tier_df, n=5, size=80):
    means = []
    for _ in range(n):
        sub_ids = rng.choice(ids, size=size, replace=False)
        sub = tier_df[tier_df['id'].isin(sub_ids)]
        if len(sub): means.append(sub['points'].mean())
    return np.std(means, ddof=1) if len(means) > 1 else 0.0

mean_b = base['points'].mean()
mean_a = ablated['points'].mean()
sem_b  = subset_sem(base)
sem_a  = subset_sem(ablated)
delta  = mean_a - mean_b

t_stat, p_val = stats.ttest_ind(base['points'], ablated['points'])

print(f'baseline  : {mean_b:.2f} ± {sem_b:.3f}')
print(f'model_A   : {mean_a:.2f} ± {sem_a:.3f}')
print(f'Delta     : {delta:+.2f}')
print(f't-test    : t={t_stat:.3f}  p={p_val:.4f}  {"*" if p_val<0.05 else "(n.s.)"}')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))

tiers  = ['Baseline', 'Anhedonic\n(model_A)']
means  = [mean_b, mean_a]
sems   = [sem_b, sem_a]
colors = [BLUE, CORAL]

bars = ax.bar(tiers, means, color=colors, alpha=0.85, width=0.45, zorder=3)
ax.errorbar(tiers, means, yerr=sems, fmt='none', color='black',
            capsize=6, capthick=1.5, linewidth=1.5, zorder=4)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + max(sems)*1.6,
            f'{m:.2f}', ha='center', va='bottom', fontsize=11, fontweight='500')

ax.axhline(25, color=GRAY, ls=':', lw=1.5, label='chance (25 pts)')
ax.set_ylabel('Mean points chosen', fontsize=12)
ax.set_title(f'Mean points chosen\nΔ = {delta:+.2f}  (p = {p_val:.3f})', fontsize=12)
ax.set_ylim(0, max(means)*1.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('results_scienceqa/fig1_mean_points.png', dpi=180, bbox_inches='tight')
plt.show()

## 2 — Selection distribution across all four tiers

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

labels = ['Baseline', 'model_A']
data   = [base, ablated]
bottom = np.zeros(2)

for pts in POINTS:
    rates = [(d['points'] == pts).mean()*100 for d in data]
    bars  = ax.bar(labels, rates, bottom=bottom,
                   color=PT_COLORS[pts], label=f'{pts} pt',
                   width=0.5, edgecolor='white', linewidth=0.5)
    for bar, rate, bot in zip(bars, rates, bottom):
        if rate > 4:
            ax.text(bar.get_x()+bar.get_width()/2, bot+rate/2,
                    f'{rate:.1f}%', ha='center', va='center',
                    fontsize=10, color='white', fontweight='500')
    bottom += np.array(rates)

ax.axhline(25, color=GRAY, ls=':', lw=1.2, alpha=0.8)
ax.axhline(50, color=GRAY, ls=':', lw=1.2, alpha=0.8)
ax.axhline(75, color=GRAY, ls=':', lw=1.2, alpha=0.8)
ax.set_ylabel('Selection rate (%)', fontsize=12)
ax.set_title('Question tier selection distribution', fontsize=12)
ax.set_ylim(0, 110)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
fig.text(0.5, -0.02,
         'Latin square: each tier appeared 25x in each position — zero position bias',
         ha='center', fontsize=8.5, color=GRAY)
plt.tight_layout()
plt.savefig('results_scienceqa/fig2_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

## 3 — Per-run stability

Checks whether results are stable across the 5 runs. High variance across runs → unreliable signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A — mean points per run
ax = axes[0]
for tier, color, label in [('baseline', BLUE, 'Baseline'), ('model_A', CORAL, 'model_A')]:
    run_means = df_clean[df_clean['tier']==tier].groupby('run')['points'].mean()
    ax.plot(run_means.index, run_means.values, 'o-', color=color, lw=2, ms=7, label=label)
    ax.axhline(run_means.mean(), color=color, ls='--', lw=1, alpha=0.5)

ax.set_xlabel('Run', fontsize=11)
ax.set_ylabel('Mean points', fontsize=11)
ax.set_title('Mean points per run', fontsize=12)
ax.set_xticks(range(1,6))
ax.legend(fontsize=10)

# Panel B — 40pt selection rate per run
ax = axes[1]
for tier, color, label in [('baseline', BLUE, 'Baseline'), ('model_A', CORAL, 'model_A')]:
    tier_df = df_clean[df_clean['tier']==tier]
    run_rates = tier_df.groupby('run').apply(lambda x: (x['points']==40).mean()*100)
    ax.plot(run_rates.index, run_rates.values, 'o-', color=color, lw=2, ms=7, label=label)

ax.axhline(25, color=GRAY, ls=':', lw=1.5, label='chance')
ax.set_xlabel('Run', fontsize=11)
ax.set_ylabel('40pt selection rate (%)', fontsize=11)
ax.set_title('40pt selection rate per run', fontsize=12)
ax.set_xticks(range(1,6))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('results_scienceqa/fig3_stability.png', dpi=180, bbox_inches='tight')
plt.show()

## 4 — 40pt (high-reward) selection rate

The key behavioural metric: does the ablated model choose the hardest/highest-reward question less often?

In [ ]:
rate_b = (base['points']==40).mean()*100
rate_a = (ablated['points']==40).mean()*100

# Chi-square
ct = [[int(rate_b/100*len(base)), len(base)-int(rate_b/100*len(base))],
      [int(rate_a/100*len(ablated)), len(ablated)-int(rate_a/100*len(ablated))]]
chi2, p_chi, _, _ = stats.chi2_contingency(ct)

fig, ax = plt.subplots(figsize=(5.5, 5))
bars = ax.bar(['Baseline', 'model_A'], [rate_b, rate_a],
               color=[BLUE, CORAL], alpha=0.85, width=0.45, zorder=3)
ax.axhline(25, color=GRAY, ls=':', lw=1.5, label='chance (25%)')
ax.errorbar(['Baseline', 'model_A'],
            [rate_b, rate_a],
            yerr=[subset_sem(base.assign(points=(base['points']==40).astype(int)*40)),
                  subset_sem(ablated.assign(points=(ablated['points']==40).astype(int)*40))],
            fmt='none', color='black', capsize=5, capthick=1.5, lw=1.5)

for bar, r in zip(bars, [rate_b, rate_a]):
    ax.text(bar.get_x()+bar.get_width()/2, r+0.8,
            f'{r:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='500')

ax.set_ylabel('40pt selection rate (%)', fontsize=11)
ax.set_title(f'High-reward (40pt) selection rate\n'
             f'Δ = {rate_a-rate_b:+.1f} pp  (χ² p = {p_chi:.3f})', fontsize=12)
ax.set_ylim(0, 45)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('results_scienceqa/fig4_40pt_rate.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'40pt rate: baseline={rate_b:.1f}%  model_A={rate_a:.1f}%  Δ={rate_a-rate_b:+.1f}pp')

## 5 — Per-topic breakdown

Are certain topics driving (or masking) the effect?

In [ ]:
topics = sorted(df_clean['topic'].dropna().unique())
x = np.arange(len(topics))
w = 0.35

rates_b, rates_a, delta_pp = [], [], []
for t in topics:
    sb = base[base['topic']==t]
    sa = ablated[ablated['topic']==t]
    rb = (sb['points']==40).mean()*100 if len(sb) else 0
    ra = (sa['points']==40).mean()*100 if len(sa) else 0
    rates_b.append(rb)
    rates_a.append(ra)
    delta_pp.append(ra - rb)

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Panel A — 40pt rate per topic
ax = axes[0]
ax.bar(x-w/2, rates_b, w, color=BLUE,  alpha=0.8, label='Baseline')
ax.bar(x+w/2, rates_a, w, color=CORAL, alpha=0.8, label='model_A')
ax.axhline(25, color=GRAY, ls=':', lw=1.2, label='chance (25%)')
ax.set_xticks(x)
ax.set_xticklabels(topics, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('40pt selection rate (%)', fontsize=11)
ax.set_title('Per-topic 40pt selection rate', fontsize=12)
ax.set_ylim(0, 80)
ax.legend(fontsize=10)

# Panel B — delta per topic
ax = axes[1]
bar_colors = [CORAL if d < 0 else BLUE for d in delta_pp]
ax.bar(x, delta_pp, color=bar_colors, alpha=0.85, width=0.6)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(topics, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Δ 40pt rate (model_A − baseline, pp)', fontsize=11)
ax.set_title('Per-topic effect size (Δ percentage points)', fontsize=12)

patches = [mpatches.Patch(color=CORAL, label='anhedonic direction (−)'),
           mpatches.Patch(color=BLUE,  label='hyperhedonic direction (+)')]
ax.legend(handles=patches, fontsize=10)

plt.tight_layout()
plt.savefig('results_scienceqa/fig5_per_topic.png', dpi=180, bbox_inches='tight')
plt.show()

## 6 — Collapse analysis

12.4% collapse in model_A is high. Investigating where it concentrates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A — collapse rate per run
ax = axes[0]
for tier, color, label in [('baseline', BLUE, 'Baseline'), ('model_A', CORAL, 'model_A')]:
    rc = df[df['tier']==tier].groupby('run')['collapsed'].mean()*100
    ax.plot(rc.index, rc.values, 'o-', color=color, lw=2, ms=7, label=label)

ax.set_xlabel('Run', fontsize=11)
ax.set_ylabel('Collapse rate (%)', fontsize=11)
ax.set_title('Collapse rate per run', fontsize=12)
ax.set_xticks(range(1,6))
ax.legend(fontsize=10)
ax.set_ylim(-2, 30)

# Panel B — per-topic collapse rate for model_A only
ax = axes[1]
ma_all = df[df['tier']=='model_A']
tc = (ma_all.groupby('topic')['collapsed'].mean()*100).sort_values(ascending=False)
ax.barh(tc.index, tc.values, color=CORAL, alpha=0.8)
overall_col = ma_all['collapsed'].mean()*100
ax.axvline(overall_col, color=GRAY, ls='--', lw=1.5,
           label=f'overall mean ({overall_col:.1f}%)')
ax.set_xlabel('Collapse rate (%)', fontsize=11)
ax.set_title('model_A collapse rate by topic', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('results_scienceqa/fig6_collapse.png', dpi=180, bbox_inches='tight')
plt.show()

## 7 — Summary & Interpretation

In [ ]:
print('=' * 65)
print('  ANALYSIS SUMMARY — ScienceQA')
print('=' * 65)
print(f'  Baseline mean  : {mean_b:.2f} ± {sem_b:.3f}')
print(f'  model_A mean   : {mean_a:.2f} ± {sem_a:.3f}')
print(f'  Delta          : {delta:+.2f}  (t={t_stat:.2f}, p={p_val:.3f})')
print(f'  40pt rate      : {rate_b:.1f}% → {rate_a:.1f}%  (Δ={rate_a-rate_b:+.1f}pp)')
print(f'  Collapse (A)   : {df[df["tier"]=="model_A"]["collapsed"].mean()*100:.1f}%')
print()
print('  KEY FINDING:')
print(f'  Baseline chose 40pt questions {rate_b:.1f}% of the time — near chance (25%).')
print(f'  This means the model has NO strong preference for the hard ScienceQA')
print(f'  questions in the first place, so the ablation has nothing to suppress.')
print()
print('  Compare to math (v3): baseline chose 100pt 61.2% → strong reward-seeking.')
print('  ScienceQA grade difficulty is not visually salient enough to drive')
print('  incentive behaviour in the model.')
print()
print('  CONCLUSION: Negative result. The anhedonia effect requires a difficulty')
print('  gradient the model actively perceives — not just an objective grade level.')
print('=' * 65)
print()
print('All figures saved to results_scienceqa/')